# 🎵 SonicBoard — Spotify Playlist Downloader
**Downloads:** MP3 320k (full metadata + album art) + `.lrc` synced lyrics  
**Output folders:**
```
Playlist Name/
  ├── MP3/   → Title - Artist.mp3
  └── LRC/   → Title - Artist.lrc
```
**Final output:** ZIP downloaded directly to your PC

---
### Before running:
1. Get free Spotify API credentials → [developer.spotify.com/dashboard](https://developer.spotify.com/dashboard)
   - Create an App → Redirect URI: `https://open.spotify.com` → check **Web API** → Save
   - ⚠️ **Feb 2026:** Your Spotify account must have an active **Premium** subscription for Dev Mode apps to work.

2. Get a free Last.fm API key → [last.fm/api/account/create](https://www.last.fm/api/account/create)
   - Used for genre tags — Spotify artist genres have been mostly empty since early 2025.
   - Optional, but without it most songs will have no genre tag.

3. Copy **Client ID**, **Client Secret**, and **Last.fm API Key** into Cell 2 below

4. Paste your **Spotify playlist URL** into Cell 3

5. `Runtime → Run all`

In [ ]:
# @title ⚙️ Cell 1 — Install dependencies (run once per session)
!pip install -q spotipy yt-dlp mutagen requests beautifulsoup4 syncedlyrics
!apt-get install -q -y ffmpeg
print('✅ All dependencies installed!')

In [ ]:
# @title 🔑 Cell 2 — Spotify API credentials

SPOTIFY_CLIENT_ID     = ""  # @param {type:"string"}
SPOTIFY_CLIENT_SECRET = ""  # @param {type:"string"}

# Last.fm API key — FREE at https://www.last.fm/api/account/create
# Used as genre fallback (Spotify genres have been largely empty since early 2025)
LASTFM_API_KEY = ""  # @param {type:"string"}

if not SPOTIFY_CLIENT_ID or not SPOTIFY_CLIENT_SECRET:
    raise ValueError('❌ Please fill in your Spotify Client ID and Secret!')
if not LASTFM_API_KEY:
    print('⚠️  No Last.fm API key — genre fallback disabled.')
    print('   Get a FREE key: https://www.last.fm/api/account/create')
else:
    print('✅ Credentials set!')


In [ ]:
# @title 🔗 Cell 3 — Paste your Spotify playlist URL

PLAYLIST_URL = "https://open.spotify.com/playlist/5f080raI4thldRpEaKd5oS?si=v2rg4XiUQgaT_-ALYMxbBQ"  # @param {type:"string"}

if not PLAYLIST_URL:
    raise ValueError('❌ Please paste a Spotify playlist URL!')
print(f'✅ Playlist URL set!')

In [ ]:
# @title 🚀 Cell 4 — Run downloader

import os, re, time, requests, unicodedata, zipfile
from pathlib import Path
from datetime import datetime
from google.colab import files

import spotipy
from spotipy.oauth2 import SpotifyOAuth
import yt_dlp
from mutagen.id3 import (
    ID3, ID3NoHeaderError,
    TIT2, TPE1, TPE2, TALB, TDRC, TRCK, TPOS,
    COMM, APIC, USLT, WOAF, TSRC, TCON
)
from bs4 import BeautifulSoup

AUDIO_FORMAT  = 'mp3'
AUDIO_BITRATE = '320'
OUTPUT_DIR    = '/content/spotify_downloads'

# ── Helpers ───────────────────────────────────────────────────

def sanitize(name):
    name = unicodedata.normalize('NFKD', name)
    name = re.sub(r'[\\/*?:"<>|]', '', name)
    return name.strip('. ')[:180] or 'unknown'

def fmt_dur(ms):
    s = ms // 1000
    return f'{s // 60}:{s % 60:02d}'

def bar(ch='─', n=58):
    print(ch * n)

# ── File naming: Title - Artist ───────────────────────────────

def make_filename(track):
    return sanitize(f"{track['title']} - {track['artist']}")

# ── Last.fm Genre Fallback ────────────────────────────────────────────────
# Spotify has been silently emptying genre data since early 2025.
# Last.fm artist.getTopTags is a reliable free fallback (no auth needed).

_LASTFM_GENRE_CACHE = {}
_LASTFM_SKIP_TAGS = {
    'seen live', 'favorites', 'favourite', 'love', 'awesome', 'classic',
    'american', 'british', 'indian', 'female vocalists', 'male vocalists',
    'all', 'spotify', 'youtube', 'beautiful', 'good',
}

def _lastfm_genres(artist_name, max_tags=3):
    """Fetch genre tags from Last.fm. Returns [] if no key configured or on error."""
    if not LASTFM_API_KEY:
        return []
    key = artist_name.lower().strip()
    if key in _LASTFM_GENRE_CACHE:
        return _LASTFM_GENRE_CACHE[key]
    try:
        r = requests.get(
            'https://ws.audioscrobbler.com/2.0/',
            params={
                'method'     : 'artist.getTopTags',
                'artist'     : artist_name,
                'autocorrect': 1,
                'api_key'    : LASTFM_API_KEY,
                'format'     : 'json',
            },
            timeout=8
        )
        if r.status_code == 200:
            tags = r.json().get('toptags', {}).get('tag', [])
            genres = [
                t['name'] for t in tags
                if t.get('name', '').lower() not in _LASTFM_SKIP_TAGS
                   and int(t.get('count', 0)) >= 10
            ][:max_tags]
            _LASTFM_GENRE_CACHE[key] = genres
            return genres
    except Exception:
        pass
    _LASTFM_GENRE_CACHE[key] = []
    return []

# ── Spotify ───────────────────────────────────────────────────

def get_tracks(url):
    sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
        client_id=SPOTIFY_CLIENT_ID,
        client_secret=SPOTIFY_CLIENT_SECRET,
        redirect_uri="https://open.spotify.com",
        scope="playlist-read-private playlist-read-collaborative",
        open_browser=False,
        show_dialog=True,
        cache_path="/content/.spotify_cache"
    ))

    # Extract plain playlist ID from URL
    playlist_id = url.strip().split("playlist/")[-1].split("?")[0]

    print('📋 Fetching playlist from Spotify...')
    pl   = sp.playlist(playlist_id, fields='name,owner')
    name = pl['name']
    print(f'   ▸ {name}  ({pl["owner"]["display_name"]})')

    tracks = []
    artist_cache = {}
    items = sp.playlist_items(playlist_id)

    while True:
        for item in items['items']:
            # Spotify API now returns 'item' key instead of 'track'
            t = item.get('track') or item.get('item')
            if not t or t.get('is_local'):
                continue
            artists = [a['name'] for a in t['artists']]
            album   = t['album']
            imgs    = album.get('images', [])
            rel     = album.get('release_date', '')

            # Safe genre fetch — handles missing 'genres' key
            artist_id = t['artists'][0]['id']
            if artist_id not in artist_cache:
                try:
                    artist_data = sp.artist(artist_id)
                    genres = artist_data.get('genres') or []
                    # Fallback to album genres if artist has none
                    if not genres:
                        # album.genres is always empty in Spotify API — never populated
                        # Use Last.fm artist.getTopTags as fallback instead
                        genres = _lastfm_genres(t['artists'][0]['name'])
                    artist_cache[artist_id] = genres
                except Exception:
                    artist_cache[artist_id] = []
            genres = artist_cache[artist_id]
            # Belt-and-suspenders: if Spotify returned empty genres, try Last.fm
            if not genres:
                genres = _lastfm_genres(t['artists'][0]['name'])
                artist_cache[artist_id] = genres

            tracks.append({
                'title'        : t['name'],
                'artist'       : ', '.join(artists),
                'artists'      : artists,
                'album_artist' : artists[0],
                'album'        : album['name'],
                'year'         : rel[:4],
                'track_number' : t.get('track_number', 0),
                'disc_number'  : t.get('disc_number', 1),
                'duration_ms'  : t.get('duration_ms', 0),
                'cover_url'    : imgs[0]['url'] if imgs else None,
                'spotify_url'  : t['external_urls'].get('spotify', ''),
                'isrc'         : t.get('external_ids', {}).get('isrc', ''),
                'genres'       : genres,
            })
        if items['next']:
            items = sp.next(items)
        else:
            break

    print(f'   ▸ {len(tracks)} tracks found\n')
    return tracks, name

# ── YouTube Download (clean, no intros) ───────────────────────
#
#  3-layer intro removal strategy:
#   1. Prefer official/VEVO/distributor uploads via search query
#   2. SponsorBlock: auto-removes marked intro/outro segments
#   3. Duration filter: rejects videos that are too long (extended mixes)

def download_audio(track, out_path):
    query = (
        f"{track['title']} {track['artists'][0]} "
        f"\"provided to youtube\" OR \"auto-generated\" OR site:vevo.com"
    )
    dur_s = track['duration_ms'] // 1000
    tmpl  = out_path.replace('.mp3', '.%(ext)s')

    opts = {
        'format'          : 'bestaudio/best',
        'outtmpl'         : tmpl,
        'quiet'           : True,
        'no_warnings'     : True,
        'noplaylist'      : True,
        'default_search'  : 'ytsearch3',
        'match_filter'    : yt_dlp.utils.match_filter_func(
            f'duration > {max(1, dur_s - 30)} & duration < {dur_s + 45}'
        ),
        'sponsorblock_remove': ['intro', 'outro', 'selfpromo', 'preview'],
        'postprocessors'  : [
            {
                'key'             : 'FFmpegExtractAudio',
                'preferredcodec'  : AUDIO_FORMAT,
                'preferredquality': AUDIO_BITRATE,
            },
            {
                'key'       : 'SponsorBlock',
                'categories': ['intro', 'outro', 'selfpromo', 'preview'],
            },
            {
                'key'                    : 'ModifyChapters',
                'remove_sponsor_segments': ['intro', 'outro', 'selfpromo', 'preview'],
            },
        ],
    }

    try:
        with yt_dlp.YoutubeDL(opts) as ydl:
            ydl.download([f'ytsearch3:{query}'])
        if os.path.exists(out_path):
            return True
    except Exception:
        pass

    # Fallback: plain search without SponsorBlock
    plain_query = f"{track['artist']} - {track['title']} audio"
    opts_plain  = {
        'format'          : 'bestaudio/best',
        'outtmpl'         : tmpl,
        'quiet'           : True,
        'no_warnings'     : True,
        'noplaylist'      : True,
        'default_search'  : 'ytsearch1',
        'match_filter'    : yt_dlp.utils.match_filter_func(
            f'duration > {max(1, dur_s - 30)} & duration < {dur_s + 45}'
        ),
        'postprocessors'  : [{
            'key'             : 'FFmpegExtractAudio',
            'preferredcodec'  : AUDIO_FORMAT,
            'preferredquality': AUDIO_BITRATE,
        }],
    }
    try:
        with yt_dlp.YoutubeDL(opts_plain) as ydl:
            ydl.download([f'ytsearch1:{plain_query}'])
        return os.path.exists(out_path)
    except Exception as e:
        print(f'         ⚠️  {e}')
        return False

# ── Metadata ──────────────────────────────────────────────────

def embed_metadata(mp3_path, track, plain_lyrics):
    try:
        try:
            tags = ID3(mp3_path)
        except ID3NoHeaderError:
            tags = ID3()
        tags.add(TIT2(encoding=3, text=track['title']))
        tags.add(TPE1(encoding=3, text=track['artist']))
        tags.add(TPE2(encoding=3, text=track['album_artist']))
        tags.add(TALB(encoding=3, text=track['album']))
        tags.add(TDRC(encoding=3, text=track['year']))
        tags.add(TRCK(encoding=3, text=str(track['track_number'])))
        tags.add(TPOS(encoding=3, text=str(track['disc_number'])))
        tags.add(TSRC(encoding=3, text=track['isrc']))
        tags.add(WOAF(url=track['spotify_url']))
        tags.add(COMM(encoding=3, lang='eng', desc='',
            text=f"Downloaded {datetime.now().strftime('%Y-%m-%d')} | ISRC:{track['isrc']}"))
        if track.get('cover_url'):
            try:
                img = requests.get(track['cover_url'], timeout=10).content
                tags.add(APIC(encoding=3, mime='image/jpeg', type=3, desc='Cover', data=img))
            except Exception:
                pass
        if plain_lyrics:
            tags.add(USLT(encoding=3, lang='eng', desc='', text=plain_lyrics))
        if track.get('genres'):
            tags.add(TCON(encoding=3, text='; '.join(track['genres'])))
        tags.save(mp3_path, v2_version=3)
    except Exception as e:
        print(f'         ⚠️  Metadata error: {e}')

# ── Lyrics ────────────────────────────────────────────────────
# Source order (best → last resort):
#  1. lrclib  — best for synced .lrc; /api/get with duration, falls back to /api/search
#               with best-match picking by duration proximity
#  2. lyrics.ovh — free REST API (no scraping, no key needed), good coverage
#  3. genius  — scraping fallback, works well for Hindi/non-English
#  4. syncedlyrics — pip package, tries Musixmatch + Apple Music + NetEase
#                    bypasses MM's anti-scrape and covers regional/Asian music

def _lrclib(title, artist, dur_s):
    try:
        # Attempt 1: exact match by duration (returns best hit if duration is close)
        r = requests.get('https://lrclib.net/api/get',
            params={'track_name': title, 'artist_name': artist, 'duration': dur_s}, timeout=10)
        if r.status_code == 200:
            d = r.json()
            if d.get('syncedLyrics', '').strip(): return d['syncedLyrics'].strip(), True
            if d.get('plainLyrics',  '').strip(): return d['plainLyrics'].strip(),  False

        # Attempt 2: search query — pick best match by duration proximity
        r2 = requests.get('https://lrclib.net/api/search',
            params={'q': f'{artist} {title}'}, timeout=10)
        if r2.status_code == 200:
            hits = r2.json()
            # Pick the hit whose duration is closest to our track's duration
            if hits:
                best = min(hits, key=lambda h: abs(h.get('duration', 0) - dur_s))
                if d.get('syncedLyrics', '').strip(): return best['syncedLyrics'].strip(), True
                if best.get('syncedLyrics', '').strip(): return best['syncedLyrics'].strip(), True
                if best.get('plainLyrics',  '').strip(): return best['plainLyrics'].strip(),  False
    except Exception:
        pass
    return None, False

def _lyrics_ovh(title, artist):
    """lyrics.ovh — free public API, no key, no scraping, decent coverage."""
    try:
        import urllib.parse
        url = f"https://api.lyrics.ovh/v1/{urllib.parse.quote(artist)}/{urllib.parse.quote(title)}"
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            text = r.json().get('lyrics', '').strip()
            if len(text) > 50:
                return text, False
    except Exception:
        pass
    return None, False

def _genius(title, artist):
    """Genius search API — works for Hindi/non-English titles."""
    try:
        r = requests.get(
            'https://genius.com/api/search/multi',
            params={'per_page': '3', 'q': f'{title} {artist}'},
            headers={'User-Agent': 'Mozilla/5.0'},
            timeout=10
        )
        if r.status_code != 200:
            return None, False
        data = r.json()
        song_url = None
        for section in data.get('response', {}).get('sections', []):
            for hit in section.get('hits', []):
                if hit.get('type') == 'song':
                    song_url = hit['result'].get('url')
                    break
            if song_url:
                break
        if not song_url:
            return None, False
        page = requests.get(song_url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        if page.status_code != 200:
            return None, False
        soup = BeautifulSoup(page.text, 'html.parser')
        for br in soup.find_all('br'):
            br.replace_with('\n')
        divs = soup.find_all('div', attrs={'data-lyrics-container': 'true'})
        if divs:
            text = '\n'.join(d.get_text() for d in divs).strip()
            if len(text) > 50:
                return text, False
    except Exception:
        pass
    return None, False

def _syncedlyrics(title, artist):
    """syncedlyrics package — tries Musixmatch, Apple Music, NetEase, etc.
    Returns synced LRC if available, otherwise None.
    Install: pip install syncedlyrics
    """
    try:
        import syncedlyrics
        lrc = syncedlyrics.search(f'{title} {artist}', allow_plain_format=True)
        if lrc and len(lrc.strip()) > 50:
            # Detect if synced (has timestamps) or plain
            is_synced = bool(re.search(r'\[\d{2}:\d{2}', lrc))
            return lrc.strip(), is_synced
    except ImportError:
        pass  # Package not installed — skip silently
    except Exception:
        pass
    return None, False

def get_lyrics(track):
    title  = track['title']
    artist = track['artists'][0]
    dur_s  = track['duration_ms'] // 1000
    for name, fn in [
        ('lrclib',       lambda: _lrclib(title, artist, dur_s)),
        ('lyrics.ovh',   lambda: _lyrics_ovh(title, artist)),
        ('genius',       lambda: _genius(title, artist)),
        ('syncedlyrics', lambda: _syncedlyrics(title, artist)),
    ]:
        try:
            text, synced = fn()
            if text: return text, synced, name
        except Exception:
            continue
    return None, False, 'none'

def build_lrc(text, is_synced, track):
    header = (
        f"[ti:{track['title']}]\n"
        f"[ar:{track['artist']}]\n"
        f"[al:{track['album']}]\n"
        f"[length:{fmt_dur(track['duration_ms'])}]\n\n"
    )
    if is_synced:
        return header + text
    lines = [l for l in text.splitlines() if l.strip()]
    out, t = [], 0
    for line in lines:
        out.append(f'[{t // 60:02d}:{t % 60:05.2f}]{line}')
        t += 3
    return header + '\n'.join(out)

# ── Main ──────────────────────────────────────────────────────

bar('═')
print('   🎵  SPOTIFY PLAYLIST DOWNLOADER  (Colab Edition)')
print('   MP3 320k + .lrc  |  Title - Artist naming  |  Clean audio')
bar('═')

tracks, pl_name = get_tracks(PLAYLIST_URL)

# ── Create separate MP3 and LRC folders ───────────────────────
base_dir = Path(OUTPUT_DIR) / sanitize(pl_name)
mp3_dir  = base_dir / 'MP3'
lrc_dir  = base_dir / 'LRC'
mp3_dir.mkdir(parents=True, exist_ok=True)
lrc_dir.mkdir(parents=True, exist_ok=True)

print(f'📁 MP3 folder : {mp3_dir}')
print(f'📁 LRC folder : {lrc_dir}\n')
bar()

done = skipped = failed = no_lrc = 0

for i, track in enumerate(tracks, 1):
    filename = make_filename(track)
    mp3_path = str(mp3_dir / f'{filename}.mp3')
    lrc_path = str(lrc_dir / f'{filename}.lrc')

    print(f"\n[{i:>3}/{len(tracks)}]  {track['title']} — {track['artist']}")
    print(f"           {track['album']}  ({track['year']})  ·  {fmt_dur(track['duration_ms'])}")
    if track['genres']:
        print(f"           🎼  {', '.join(track['genres'])}")

    if os.path.exists(mp3_path) and os.path.exists(lrc_path):
        print('           ✅  Already exists — skipping.')
        skipped += 1
        continue

    # ── Download audio
    if not os.path.exists(mp3_path):
        print('           ⬇️   Downloading (clean audio, removing intros)...')
        if not download_audio(track, mp3_path):
            print('           ❌  Download failed.')
            failed += 1
            continue
    else:
        print('           ✅  Audio already present.')

    plain_for_embed = None

    # ── Fetch lyrics
    if not os.path.exists(lrc_path):
        print('           📝  Fetching lyrics...')
        text, is_synced, source = get_lyrics(track)
        if text:
            kind = 'synced ✨' if is_synced else 'plain → .lrc'
            print(f'           ✅  {source} ({kind})')
            plain_for_embed = text
            with open(lrc_path, 'w', encoding='utf-8') as f:
                f.write(build_lrc(text, is_synced, track))
        else:
            print('           ⚠️   No lyrics found — creating placeholder .lrc.')
            dur_str = fmt_dur(track['duration_ms'])
            funny_lines = (
        "[00:08.00] 🎧 Welcome to the \"No Lyrics\" experience\n"
        "[00:14.00] Sit back, relax, and pretend you know the words\n"
        "[00:20.00] Our lyricist is currently on a chai break ☕\n"
        "[00:26.00] He said he'll be back \"in 2 minutes\"...\n"
        "[00:32.00] That was 3 hours ago 😐\n"
        "[00:38.00] Meanwhile, the singer is absolutely vibing 🎤\n"
        "[00:44.00] And you? You're just guessing the lyrics 😄\n"
        "[00:50.00] Was that \"dil\" or \"deal\" or \"dhill\"? 🤔\n"
        "[00:56.00] Congratulations, you just created your own remix 🎶\n"
        "[01:02.00] Loading lyrics... █▒▒▒▒▒▒▒▒▒ 10%\n"
        "[01:08.00] Still loading... ███▒▒▒▒▒▒▒ 30%\n"
        "[01:14.00] Almost there... █████▒▒▒▒▒ 50%\n"
        "[01:20.00] Just kidding 😆 nothing is loading\n"
        "[01:26.00] But hey, you're still listening!\n"
        "[01:32.00] Imagine if you actually knew the lyrics 😏\n"
        "[01:38.00] Life would be too easy then\n"
        "[01:44.00] Tip: Just hum confidently\n"
        "[01:50.00] No one knows you're wrong 😎\n"
        "[01:56.00] 🎶 La la la... universal lyrics unlocked\n"
        "[02:02.00] If you're still here, respect 🫡\n"
        "[02:08.00] Lyrics are still missing btw\n"
        "[02:14.00] Okay fine... last update:\n"
        "[02:20.00] Lyrics developer has gone offline 🚫\n"
        "[02:26.00] Please try again later...\n"
        "[02:32.00] Or just enjoy the music ❤️\n"
        "[02:48.48]\n"
    )
            placeholder_lrc = (
                f"[ti:{track['title']}]\n"
                f"[ar:{track['artist']}]\n"
                f"[al:{track['album']}]\n"
                f"[length:{dur_str}]\n"
                f"\n"
                + funny_lines
            )
            with open(lrc_path, 'w', encoding='utf-8') as f:
                f.write(placeholder_lrc)
            plain_for_embed = funny_lines  # Don't embed placeholder text into MP3
            no_lrc += 1
    else:
        print('           ✅  LRC already present.')
        with open(lrc_path, encoding='utf-8') as f:
            plain_for_embed = f.read()

    # ── Embed metadata
    print('           🏷️   Embedding metadata & album art...')
    embed_metadata(mp3_path, track, plain_for_embed)
    print('           🎉  Done!')
    done += 1
    time.sleep(0.5)

# ── Summary
print()
bar('═')
print(f'   ✅  Downloaded  : {done}')
print(f'   ⏭️   Skipped     : {skipped}')
print(f'   ❌  Failed      : {failed}')
print(f'   🔇  No lyrics   : {no_lrc}')
bar('═')

# ── ZIP and download
print('\n📦  Zipping files...')
zip_name = f'/content/{sanitize(pl_name)}.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in base_dir.rglob('*'):
        zf.write(f, f.relative_to(base_dir))

size_mb = os.path.getsize(zip_name) / 1024 / 1024
print(f'✅  ZIP ready: {sanitize(pl_name)}.zip  ({size_mb:.1f} MB)')
print('⬇️   Starting download to your PC...\n')
files.download(zip_name)